## Nikola Stojanovic

## Assignment 3: Spark

## Learning Outcomes

In this assignment, you will do the following:

* Import a dataset into the Databricks Spark environment

* Create tables for the data imported

* Perform basic data analysis using transformations and Spark SQL

**Setup Instructions**

1. Read the article:
> https://databricks.com/blog/2016/07/14/a-tale-of-three-apache-spark-apis-rdds-dataframes-and-datasets.html (links to an external site)


2. Download Data: You an either download the JSON file from the Assignment 3 home page in Learn under 'EXPECTATIONS AND ASSIGNMENT FILES' or you can download the file from Github to your computer: 
> https://github.com/dmatrix/examples/blob/master/spark/databricks/notebooks/py/data/iot_devices.json (links to an external site) Click on the raw data source -> right click -> click save as. The file will download locally and now you can import it to Databricks. **NOTE**: 

3. Import files: How to import your downloaded files (from step 3) to your Databricks cluster: https://www.projectpro.io/recipes/create-dataframe-from-json-file-read-data-from-dbfs-and-write-into-dbfs (links to an external site).


4. Import the notebook below.  There is some data exploration already done in this notebook for your reference. For details on how to import the notebook above see: https://docs.databricks.com/user-guide/notebooks/index.html (links to an external site).
> https://databricks-prod-cloudfront.cloud.databricks.com/public/4027ec902e239c93eaaa8714f173bcfc/5915990090493625/411609171004360/6085673883631125/latest.html


5. Run it. **NOTE**: Don't forget to create a cluster and attach the imported notebook to it (left upper corner: button `detached`) before trying to run it.

**Questions** (12 marks)

1. Explain the main differences between RDDs, Dataframes and Datasets (4 marks)


2. Answer the following questions:

   2.1 How many sensor pads are reported to be from Poland (2 marks) 
   
   2.2 How many different LCDs (distinct colors) are present in the dataset (2 marks)  
   
   2.3 Find 5 countries that have the largest number of MAC devices used (2 marks)
   
   2.4 Propose and try an interesting statistical test or machine learning model you could use to gain insight from this dataset. Note, you don't have to use Machine Learning for this question. You can apply any analysis to the data even using SparkSQL, Python visualization libraries to analyze the data. Another example cloud be to apply correlation functions or other Spark functions to analyze the data. (2 marks)
   

**NOTE**: You may use MLLib in 2.4: https://spark.apache.org/docs/latest/ml-guide.html. Marks are awarded for the idea and implementation of the test/ML model.  

Please submit the **published** notebook link in a word/pdf document. Do not submit HTML, Jupyter notebook, or archive (DBC) formats.  


### # 1. Explain the main differences between RDDs, Dataframes and Datasets (4 marks)

RDDs (Resilient Distributed Datasets) are Spark's lowest-level abstraction: distributed collections of JVM objects with no schema.  Transformations (map, filter) run without any query optimization, and fault tolerance comes from lineage tracking

Dataframes organize data into named columns with a schema,this allows Spark's Catalyst optimizer to optimize execution such as (predicate pushdown, column pruning, code generation). This make them much faster than RDDs, but are not type safe. Errors like referencing a bad column only appear at runtime. It is available for Scala, Java, Python, and R. 

Datasets extend Dataframes by adding compile-time type safety. A DataSet is a typed collection like (Dataset[T]), where a DataFrame is technically just
(DataFrame[Row]). Datasets use encoders to convert between JVM objects and Spark's internal binary format, avoiding Java serialization costs while preserving type checking. However, Datasets are only available in Scala and Java, since Python and R ar dynamically types and gain no benefit from compile-time type enforcement. 


In [0]:
#imports
#from pyspark.sql.functions import col, asc, count
from pyspark.sql import functions as F


# read the json file and create the dataframe

file_location = "/Volumes/workspace/default/assignment3"

# The applied options are for CSV files. For other file types, these will be ignored.
'''df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", first_row_is_header) \
  .option("sep", delimiter) \
  .load(file_location)
'''
df = spark.read.json(file_location)
raw_count = df.count()
display(raw_count)

In [0]:
display(df.take(5))

In [0]:
# Create a view or table (Temporary View)
df.createOrReplaceTempView( "iot_devices")

In [0]:
%sql 
--Showing the data from the temo view
SELECT * FROM iot_devices LIMIT 10;

In [0]:
# Permanent Table, saved survives across multiple sessions
df.write.mode("overwrite").saveAsTable("iot_devices_table")
print("Table 'iot_devices_table' created")

In [0]:
#Running SQL, return dataframe back 
results_df = spark.sql( '''
    
    SELECT cn, AVG(battery_level) as avg_battery FROM iot_devices_table
    GROUP BY cn
    
''')

results_df.show()


In [0]:
step1 = spark.sql('''
    SELECT * FROM iot_devices_table WHERE battery_level > 5
''')
#display(step1)


step2 = step1.withColumn("battery_ok", step1.battery_level > 3)

display(step2)

In [0]:
df_filtered = df.filter(df.battery_level < 5)
df_filtered.groupBy("cn").count().alias("Count").orderBy("Count", ascending=False).show()

#.alias("MACDevices_Used")).orderBy("MACDevices_Used", ascending=False).take(5)

### ## Data exploration

In [0]:
df.printSchema()

In [0]:
display(df)

  2.1 How many sensor pads are reported to be from Poland (2 marks)


In [0]:

#2.1 How many sensor pads are reported to be from Poland (2 marks)

sensorPad_Poland_df = spark.sql('''
        SELECT COUNT (*) AS Num_of_SensorPads 
        FROM iot_devices_table
        WHERE cn = "Poland" AND device_name LIKE 
        "sensor-pad%"              
''')
display(sensorPad_Poland_df)


#sensorPad_Poland_count = df.filter(("cn") == "Poland").filter(col("device_name").like("%sensor-pad%")).count()
#display(sensorPad_Poland_count) 

''' There are 1413 sensor pads from Poland in this dataset.'''


2.2 How many different LCDs (distinct colors) are present in the dataset (2 marks)


In [0]:
#2.2 How many different LCDs (distinct colors) are present in the dataset (2 marks)

common_lcd_colours = spark.sql('''
            SELECT DISTINCT lcd AS lcd_colours
            FROM iot_devices_table;            
''')

display(common_lcd_colours)


''' There are 3 different lcd colours in the dataset '''

lcd_colours_count = spark.sql('''
            SELECT COUNT(DISTINCT lcd )AS lcd_colours
            FROM iot_devices_table;            
''')

display(lcd_colours_count)


#lcd_colours = df.filter(col("lcd").isNotNull()).select("lcd").distinct()
#display(lcd_colours)


 2.3 Find 5 countries that have the largest number of MAC devices used (2 marks)

In [0]:
# 2.3 Find 5 countries that have the largest number of MAC devices used (2 marks)

countries_MACdevice = spark.sql ('''
            SELECT cn, COUNT(device_name) AS Num_of_MACDevices
            FROM iot_devices_table
            WHERE device_name LIKE "device-mac%"
            GROUP BY cn
            ORDER BY Num_of_MACDevices DESC
            LIMIT 5
''')

#countries_MACdevice = df.select("cn","device_name").filter(F.col("device_name").like("device-mac%")).groupBy("cn").agg(F.count("device_name").alias("MACDevices_Used")).orderBy("MACDevices_Used", ascending=False).take(5)
display(countries_MACdevice)


2.4 Propose and try an interesting statistical test or machine learning model you could use to gain insight from this dataset. Note, you don't have to use Machine Learning for this question. You can apply any analysis to the data even using SparkSQL, Python visualization libraries to analyze the data. Another example cloud be to apply correlation functions or other Spark functions to analyze the data. (2 marks)

In [0]:
#from pyspark.sql.functions import corr
display(df.select(F.corr("temp", "c02_level")))
display(df.select(corr("humidity", "battery_level")))

In [0]:
sample_pdf = df.select("temp","c02_level").sample(0.01).toPandas()

import matplotlib.pyplot as plt
plt.scatter(sample_pdf["temp"], sample_pdf['c02_level'], alpha=0.3)
plt.xlabel("Temperature")
plt.ylabel("CO2 Level")
plt.title("Temp vs CO2 Level")
plt.show()


In [0]:
#2.4 
from pyspark.sql.functions import avg, round, desc

#calculate average environmental conditions for each country

country_insights = df.groupBy("cn").agg(
    round (avg("temp"),2).alias("Avg_Temp"),
    round(avg("humidity"),2).alias("Avg_Humidity"),
    round(avg("c02_level"),2).alias("Avg_C02")
).orderBy(desc("Avg_C02"))

display(country_insights)





In [0]:
from pyspark.sql.functions import avg, round, desc
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Aggregate the data using Spark
country_insights = df.groupBy("cn") \
                     .agg(round(avg("c02_level"), 2).alias("Avg_CO2")) \
                     .orderBy(desc("Avg_CO2"))

# 2. Convert the top 10 results to a Pandas DataFrame for plotting
pandas_df = country_insights.limit(10).toPandas()

# 3. Create the visualization plot
plt.figure(figsize=(12, 6))
sns.barplot(x="Avg_CO2", y="cn", data=pandas_df, palette="viridis")

# 4. Customize chart details
plt.title("Top 10 Countries by Average Device CO2 Levels", fontsize=14, fontweight='bold')
plt.xlabel("Average CO2 Level (ppm)", fontsize=12)
plt.ylabel("Country", fontsize=12)
plt.tight_layout()

# 5. Show the plot inside the notebook
plt.show()

### Question 2.4 Analysis & Interpretation

**Proposed Analysis:** Global Geographic Environmental Profiling using Spark SQL Aggregations.

**Findings & Insight:**
By grouping the dataset by country (`cn`) and calculating the average temperature, humidity, and carbon dioxide levels, we move past a single global average to see localized environmental baselines. Sorting the output by the highest average CO₂ level allows us to immediately identify countries where hardware assets are deployed in poorly ventilated spaces or dense industrial environments.

**Conclusion:**
This statistical profile provides direct operational value. Instead of treats all devices globally as identical, it allows maintenance teams to set localized threshold baselines. For example, countries displaying an abnormally high baseline for average CO₂ or temperature can be flagged for immediate HVAC audits or prioritized for sensor fleet maintenance.

In [0]:
%sql
 select cca3, count(distinct device_id) as device_id from iot_devices_table group by cca3 order by device_id desc limit 100

Databricks visualization. Run in Databricks to view.

In [0]:
# Mapping four fields- temp, device_name, device_id, cca3 
dfTempMap = df.select("temp", "device_name", "device_id", "cca3").filter(F.col("temp")> 25).take(7)
display(dfTempMap)



In [0]:
#Select all countries' devices with high-levels of C02 and group by cca3 and order by device_ids

display(df.select("cca3", "cn", "device_id").filter(col("c02_level")> 1).groupBy("cca3", "cn").agg(count("device_id").alias("device_ids")).orderBy("device_ids").take(5))


In [0]:
battLevelMore6 = df.select("battery_level", "c02_level", "device_name").filter(col("battery_level")>6).sort(col("c02_level"))
display(battLevelMore6)

In [0]:
#Correlation between temp and c02 level
display(df.select("humidity", "temp").stat.corr("humidity", "temp"))

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

pdf = df.select("temp", "c02_level").toPandas()

plt.figure(figsize=(8, 5))
plt.scatter(pdf["temp"], pdf["c02_level"],alpha=0.5)
plt.xlabel("Temperature (°C)")
plt.ylabel("CO₂ Level")
plt.title("Scatter Plot of Temperature vs CO₂ Level")
plt.show()


In [0]:
#Average c02 level by country
display(df.groupBy("cn").avg("c02_level").orderBy(("avg(c02_level)"), ascending=False).take(5))

In [0]:
%sql
    
select cca3, count(device_id) as device_id from iot_devices_table where lcd = 'red' group by cca3 order by device_id desc limit 100

Databricks visualization. Run in Databricks to view.

In [0]:
# batteries that need replacements

#df.select("cn","device_name").filter(col("device_name").like("device-mac%")).groupBy("cn").agg(count#("device_name").alias("MACDevices_Used")).orderBy("MACDevices_Used", ascending=False).take(5)

display(df.select("cn", "device_id").filter(col("battery_level")==0).groupBy("cn").agg(count("device_id").alias("Devices_with_0_battery")).orderBy("Devices_with_0_battery", ascending=False).take(5))


#select cca3, count(distinct device_id) as device_id from iot_device_data where battery_level == 0 group by cca3 order by device_id desc limit 100

Databricks visualization. Run in Databricks to view.

In [0]:
df.describe("c02_level").show()

In [0]:
from pyspark.sql.functions import avg
display(df.groupBy("cn").agg((avg("c02_level").alias("avg_c02Level)"))).orderBy("avg_c02Level)", ascending= False).show())

In [0]:
#Option 3: Hypothesis Testing 
# Does battery level differ significantly between two countries/ regions?





